## DermaVision AI — Silver 2.1: Single-Image Inference + Web App
## **Teammates: Kishore, Ellie Lansdown, Ryan Bennett, Grace Callahan & Stephanie Furst**
## **Team: Group 7 | Modern AI for Business | MSBA | April 2026**

**What Silver 2.1 adds on top of Silver 2.0:**

| Addition | Description |
|----------|-------------|
| Single-image inference | Load the three saved checkpoints and run the weighted ensemble on any image |
| FastAPI backend | `webapp/main.py` — REST endpoint that accepts an uploaded image, returns class probabilities |
| HTML frontend | `webapp/templates/index.html` — drag-and-drop upload + probability bar display |
| Colab launcher | Section 5 — spin up the web app inside Colab with an ngrok public URL |

**Silver 2.0 is unchanged.** All training code stays in `Silver-model-2.0.ipynb`.
This notebook only loads the checkpoints produced by that run.

### Silver 2.0 Results (reference)
| Model | Bal.Acc | AUROC | Mel Recall | BCC Recall |
|-------|---------|-------|-----------|------------|
| EfficientNetB0 | 0.372 | 0.869 | 0.531 | 0.043 |
| SwinV2 | 0.713 | 0.957 | 0.761 | 0.748 |
| BiomedCLIP | 0.602 | 0.930 | 0.797 | 0.653 |
| **Weighted Ensemble** | **0.721** | **0.957** | **0.788** | **0.733** |

## 1. Installs

In [ ]:
# Run once. Restart runtime if prompted by Colab after TF install.
!pip install -q fastapi uvicorn[standard] python-multipart pyngrok nest_asyncio
!pip install -q timm open_clip_torch
print('Installs complete.')

## 2. Imports & Constants

All constants mirror `Silver-model-2.0.ipynb` exactly so the loaded checkpoints are compatible.

In [ ]:
import os, io, warnings
import numpy as np
from PIL import Image
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import timm
import open_clip

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'TF version: {tf.__version__}  |  PyTorch: {torch.__version__}')

# ── Classes (9-class canonical label space from Silver 2.0) ──
LABEL_NAMES = ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc', 'scc', 'unk']
LABEL2IDX   = {l: i for i, l in enumerate(LABEL_NAMES)}
N_CLASSES   = len(LABEL_NAMES)

LABEL_FULL = {
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevi',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis / IEC',
    'bkl':   'Benign Keratosis-Like',
    'df':    'Dermatofibroma',
    'vasc':  'Vascular Lesion',
    'scc':   'Squamous Cell Carcinoma',
    'unk':   'Unknown / Other',
}
RISK_TIER = {
    'mel': 'HIGH', 'bcc': 'HIGH', 'scc': 'HIGH',
    'akiec': 'MODERATE',
    'nv': 'low', 'bkl': 'low', 'df': 'low', 'vasc': 'low', 'unk': 'low',
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Constants ready.')

## 3. Checkpoint Paths & Ensemble Config

Update `CKPT_DIR` to wherever Google Drive mounted your Silver 2.0 checkpoints.  
Update `W_EFF / W_SWIN / W_CLIP` and `T_SWIN / T_CLIP` with the
**actual Nelder-Mead weights and temperature scalars** printed at the end of Silver-model-2.0.ipynb.

In [ ]:
# ── Mount Drive (Colab only) ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Checkpoint paths (match Silver-model-2.0.ipynb CKPT_DIR) ─────────────────
CKPT_DIR  = '/content/drive/MyDrive/SILVER FOLDER/checkpoints'
EFF_CKPT  = f'{CKPT_DIR}/effnet_silver2_best.keras'
SWIN_CKPT = f'{CKPT_DIR}/swinv2_silver2_best.pt'
CLIP_CKPT = f'{CKPT_DIR}/biomedclip_silver2_best.pt'

for path in [EFF_CKPT, SWIN_CKPT, CLIP_CKPT]:
    status = '✓ found' if os.path.exists(path) else '✗ MISSING'
    print(f'{status}  {path}')

# ── Ensemble weights ──────────────────────────────────────────────────────────
# Replace these with the exact values printed by Silver-model-2.0.ipynb Cell 52.
# The values below are the Nelder-Mead starting point.
W_EFF  = 0.20
W_SWIN = 0.50
W_CLIP = 0.30

# ── Temperature scalars ───────────────────────────────────────────────────────
# Replace with T values printed by Silver-model-2.0.ipynb Cell 46.
T_SWIN = 1.50
T_CLIP = 1.50

total = W_EFF + W_SWIN + W_CLIP
print(f'\nEnsemble weights: EfficientNetB0={W_EFF/total:.2f}  SwinV2={W_SWIN/total:.2f}  BiomedCLIP={W_CLIP/total:.2f}')
print(f'Temperature:      SwinV2={T_SWIN}  BiomedCLIP={T_CLIP}')

## 4. Load Checkpoints & Define Single-Image Inference

In [ ]:
# ── FocalLoss (must match Silver-model-2.0 definition for .keras deserialization) ──
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=None, name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred   = tf.clip_by_value(y_pred, 1e-8, 1.0 - 1e-8)
        ce       = -y_true * tf.math.log(y_pred)
        focal_wt = tf.pow(1.0 - y_pred, self.gamma)
        fl       = focal_wt * ce
        if self.alpha is not None:
            fl = self.alpha * fl
        return tf.reduce_mean(tf.reduce_sum(fl, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma})
        return cfg

print('FocalLoss defined.')

In [ ]:
# ── EfficientNetB0 ────────────────────────────────────────────────────────────
print('Loading EfficientNetB0...')
eff_model = tf.keras.models.load_model(EFF_CKPT, custom_objects={'FocalLoss': FocalLoss})
print(f'  ✓ EfficientNetB0 loaded  ({eff_model.count_params():,} params)')

In [ ]:
# ── SwinV2-tiny (architecture must match Silver-model-2.0.ipynb Cell 37) ──────
print('Loading SwinV2...')

swin_backbone = timm.create_model('swinv2_tiny_window8_256', pretrained=False,
                                   num_classes=0, global_pool='avg')
embed_dim_swin = swin_backbone.num_features

swin_head = nn.Sequential(
    nn.Linear(embed_dim_swin, 512), nn.ReLU(), nn.Dropout(0.4),
    nn.Linear(512, 256),            nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, N_CLASSES)
).to(DEVICE)

class SwinClassifier(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head     = head
    def forward(self, x):
        return self.head(self.backbone(x))

swin_model = SwinClassifier(swin_backbone, swin_head).to(DEVICE)
swin_model.load_state_dict(torch.load(SWIN_CKPT, map_location=DEVICE))
swin_model.eval()
print(f'  ✓ SwinV2 loaded  ({sum(p.numel() for p in swin_model.parameters()):,} params)')

In [ ]:
# ── BiomedCLIP (architecture must match Silver-model-2.0.ipynb Cell 41) ───────
print('Loading BiomedCLIP...')

clip_raw, _, clip_val_transform = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
vision_encoder = clip_raw.visual.to(DEVICE)

with torch.no_grad():
    dummy          = torch.randn(1, 3, 224, 224).to(DEVICE)
    embed_dim_clip = vision_encoder(dummy).shape[1]

class BiomedCLIPClassifier(nn.Module):
    def __init__(self, encoder, embed_dim, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256),       nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    def forward(self, x):
        return self.head(self.encoder(x))

clip_model = BiomedCLIPClassifier(vision_encoder, embed_dim_clip, N_CLASSES).to(DEVICE)
clip_model.load_state_dict(torch.load(CLIP_CKPT, map_location=DEVICE))
clip_model.eval()
print(f'  ✓ BiomedCLIP loaded  ({sum(p.numel() for p in clip_model.parameters()):,} params)')

In [ ]:
# ── Single-image inference function ──────────────────────────────────────────
# Preprocessing transforms (match Silver-model-2.0.ipynb)
swin_val_tf = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def predict_image(img_path: str, verbose: bool = True) -> dict:
    """
    Run the Silver 2.1 weighted ensemble on a single image.

    Parameters
    ----------
    img_path : str
        Path to image file (JPEG, PNG, etc.).
    verbose : bool
        Print a formatted result table.

    Returns
    -------
    dict with keys: top_class, top_full, risk_tier, ensemble, efficientnet, swinv2, biomedclip
    """
    img = Image.open(img_path).convert('RGB')

    # EfficientNetB0
    arr      = eff_preprocess(np.expand_dims(np.array(img.resize((224, 224)), dtype=np.float32), 0))
    prob_eff = eff_model.predict(arr, verbose=0)[0]  # shape [9]

    # SwinV2 + temperature scaling
    x_swin = swin_val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob_swin = F.softmax(swin_model(x_swin) / T_SWIN, dim=1).cpu().numpy()[0]

    # BiomedCLIP + temperature scaling
    x_clip = clip_val_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob_clip = F.softmax(clip_model(x_clip) / T_CLIP, dim=1).cpu().numpy()[0]

    # Weighted ensemble
    total    = W_EFF + W_SWIN + W_CLIP
    ensemble = (W_EFF/total)*prob_eff + (W_SWIN/total)*prob_swin + (W_CLIP/total)*prob_clip

    top_idx   = int(ensemble.argmax())
    top_label = LABEL_NAMES[top_idx]

    result = {
        'top_class':    top_label,
        'top_full':     LABEL_FULL[top_label],
        'risk_tier':    RISK_TIER[top_label],
        'ensemble':     dict(zip(LABEL_NAMES, ensemble.tolist())),
        'efficientnet': dict(zip(LABEL_NAMES, prob_eff.tolist())),
        'swinv2':       dict(zip(LABEL_NAMES, prob_swin.tolist())),
        'biomedclip':   dict(zip(LABEL_NAMES, prob_clip.tolist())),
    }

    if verbose:
        print(f'\n=== Silver 2.1 Ensemble Prediction ===')
        print(f'  Top class   : {top_label.upper()}  —  {LABEL_FULL[top_label]}')
        print(f'  Risk tier   : {RISK_TIER[top_label]}')
        print(f'  Confidence  : {ensemble[top_idx]*100:.1f}%')
        print()
        header = f'  {"Class":<8} {"Ensemble":>9} {"EffNet":>9} {"SwinV2":>9} {"CLIP":>9}'
        print(header)
        print('  ' + '-' * 50)
        order = sorted(range(N_CLASSES), key=lambda i: -ensemble[i])
        for i in order:
            lbl = LABEL_NAMES[i]
            print(f'  {lbl:<8} {ensemble[i]:>8.1%} {prob_eff[i]:>8.1%} {prob_swin[i]:>8.1%} {prob_clip[i]:>8.1%}')

    return result

print('predict_image() ready.')

### 4b. Quick Inference Demo

Replace `TEST_IMAGE` with a path to any image on Drive or Colab local storage.

In [ ]:
import matplotlib.pyplot as plt

TEST_IMAGE = '/content/drive/MyDrive/SILVER FOLDER/sample_test_image.jpg'  # ← change me

if os.path.exists(TEST_IMAGE):
    result = predict_image(TEST_IMAGE)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: image
    axes[0].imshow(Image.open(TEST_IMAGE).convert('RGB'))
    axes[0].set_title(f'Input: {os.path.basename(TEST_IMAGE)}')
    axes[0].axis('off')

    # Right: ensemble probability bars
    ens = result['ensemble']
    labels_sorted = sorted(ens, key=lambda l: -ens[l])
    probs_sorted  = [ens[l] for l in labels_sorted]
    colors        = ['#fc8181' if RISK_TIER[l] == 'HIGH'
                     else '#f6ad55' if RISK_TIER[l] == 'MODERATE'
                     else '#68d391' for l in labels_sorted]

    axes[1].barh(labels_sorted[::-1], probs_sorted[::-1], color=colors[::-1])
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Silver 2.1 Ensemble — Class Probabilities')
    for i, (lbl, p) in enumerate(zip(labels_sorted[::-1], probs_sorted[::-1])):
        axes[1].text(p + 0.01, i, f'{p:.1%}', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print(f'TEST_IMAGE not found: {TEST_IMAGE}')
    print('Update the path above and re-run.')

## 5. Launch the Web App (FastAPI + ngrok)

This section writes the `webapp/` files to Colab local storage and serves the app
at a public ngrok URL you can open in any browser.

**Prerequisites:**
1. Get a free ngrok auth token at https://dashboard.ngrok.com/signup
2. Paste it into `NGROK_TOKEN` below.

In [ ]:
# Write webapp files to Colab local storage
# (These files live in the GitHub repo under webapp/ — copy them here or clone the repo)

import subprocess

# Clone the repo so we have main.py and the HTML template
if not os.path.exists('/content/dermavision-ai'):
    subprocess.run(['git', 'clone',
                    'https://github.com/rbennett16722-dot/dermavision-ai.git',
                    '/content/dermavision-ai'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', '/content/dermavision-ai', 'pull'], check=True)
    print('Repo updated.')

WEBAPP_DIR = '/content/dermavision-ai/webapp'
print(f'webapp path: {WEBAPP_DIR}')
print('Files:', os.listdir(WEBAPP_DIR))

In [ ]:
# ── Inject checkpoint paths & weights into webapp environment variables ────────
# The FastAPI app reads these at startup (see webapp/main.py).

os.environ['CKPT_DIR']  = CKPT_DIR
os.environ['EFF_CKPT']  = EFF_CKPT
os.environ['SWIN_CKPT'] = SWIN_CKPT
os.environ['CLIP_CKPT'] = CLIP_CKPT
os.environ['W_EFF']     = str(W_EFF)
os.environ['W_SWIN']    = str(W_SWIN)
os.environ['W_CLIP']    = str(W_CLIP)
os.environ['T_SWIN']    = str(T_SWIN)
os.environ['T_CLIP']    = str(T_CLIP)

print('Environment variables set for FastAPI app.')

In [ ]:
# ── Launch FastAPI via uvicorn + expose with ngrok ────────────────────────────
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf

NGROK_TOKEN = ''   # ← paste your ngrok auth token here
PORT        = 8000

assert NGROK_TOKEN, 'Set NGROK_TOKEN before running this cell.'
conf.get_default().auth_token = NGROK_TOKEN

# Add the webapp directory to Python path so uvicorn finds main.py
import sys
if WEBAPP_DIR not in sys.path:
    sys.path.insert(0, WEBAPP_DIR)

# Allow uvicorn to run inside Jupyter's event loop
nest_asyncio.apply()

# Start ngrok tunnel
public_url = ngrok.connect(PORT).public_url
print(f'\n=== DermaVision AI is live ===')
print(f'  Open this URL in your browser: {public_url}')
print(f'  (Ctrl+C or stop the cell to shut down)\n')

# Run uvicorn in a background thread so the cell stays interactive
def run_server():
    uvicorn.run('main:app', host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

## 6. Silver 2.1 Summary

| Component | File | Description |
|-----------|------|-------------|
| Backend | `webapp/main.py` | FastAPI app — loads checkpoints, `/predict` endpoint |
| Frontend | `webapp/templates/index.html` | Drag-and-drop upload, probability bars, per-model accordion |
| Dependencies | `webapp/requirements.txt` | pip install list for local runs |
| This notebook | `Silver-model-2.1.ipynb` | Inference demo + Colab launcher |

### Running locally (outside Colab)
```bash
cd webapp
pip install -r requirements.txt

# Set checkpoint paths
export CKPT_DIR=/path/to/checkpoints
export W_SWIN=0.50   # update with your actual optimised weights
export T_SWIN=1.42   # update with your actual T values

uvicorn main:app --reload --port 8000
# open http://localhost:8000
```

### What the frontend shows
- **Top prediction banner** — class name, risk tier (HIGH / MODERATE / low), confidence
- **Probability bars** — all 9 classes, colour-coded by risk, sorted by ensemble score
- **Per-model accordion** — top-5 classes for each of the three models individually
- **Ensemble weights** — displayed so users know how the three models contributed